<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-05-bigquery-ml/lesson-5.4-registry/practice/GCP_Capstone_5.4_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 5.4 — BigQuery to Vertex AI

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup

Authenticate with Application Default Credentials (no API keys) and initialise the BigQuery client. Run this cell first — every exercise below depends on `PROJECT_ID`, `client`, and the `run_query` / `run_ddl` helpers.

> **Prerequisite:** run Lesson 5.1 first so that `rag_data.document_features` exists in your project (BigQuery tables persist per project).

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE
LOCATION = 'us-central1'           # course region (asia-south1 for India prod)
USD_INR = 85                       # for any INR cost display

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

# Ensure this module's datasets exist (idempotent -- BigQuery never auto-creates them).
# NOTE: cells that read rag_data.document_features need Lesson 5.1 run first.
for _ds in ('rag_data', 'ml_models'):
    _d = bigquery.Dataset(f'{PROJECT_ID}.{_ds}'); _d.location = 'US'
    client.create_dataset(_d, exists_ok=True)

def run_query(sql):
    return client.query(sql).to_dataframe()

def run_ddl(sql):
    job = client.query(sql)
    job.result()
    print(f'Done: {job.num_dml_affected_rows or "OK"}')

print(f'Connected to {PROJECT_ID}')

## Exercise 1: Register a Model

**Difficulty:** Easy

Train any model with `model_registry='vertex_ai'`. Verify in Vertex AI Console.

1. Add `model_registry='vertex_ai'` to OPTIONS
2. Set `vertex_ai_model_id`
3. Check Vertex AI Console → Model Registry

In [ ]:
# Train + register in one step.
# The model_registry + vertex_ai_model_id OPTIONS auto-publish it to Vertex AI.
run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.doc_classifier_v2`
OPTIONS (
  model_type = 'BOOSTED_TREE_CLASSIFIER',
  input_label_cols = ['document_type'],
  model_registry = 'vertex_ai',
  vertex_ai_model_id = 'documind_doc_classifier',
  vertex_ai_model_version_aliases = ['v2', 'latest'],
  max_iterations = 50,
  enable_global_explain = TRUE
) AS
SELECT page_count, chunk_count, total_word_count,
       file_size_mb, content_type, avg_chunk_size, document_type
FROM `{PROJECT_ID}.rag_data.document_features`
''')
print('Model trained and registered in Vertex AI')
print('Check: Vertex AI Console -> Model Registry (source = "BigQuery ML")')

Confirm it evaluated cleanly and inspect Shapley feature importance while you are here.

In [ ]:
print('=== Classification Metrics ===')
print(run_query(f'SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.ml_models.doc_classifier_v2`)'))

print('\n=== Feature Importance (Shapley) ===')
print(run_query(f'SELECT * FROM ML.GLOBAL_EXPLAIN(MODEL `{PROJECT_ID}.ml_models.doc_classifier_v2`)'))

## Exercise 2: EXPORT MODEL to GCS

**Difficulty:** Easy

Export the classifier to a GCS bucket. List exported files.

1. EXPORT MODEL with URI to GCS
2. Verify bucket and region match
3. List files with `gsutil ls`

In [ ]:
# Export model artifacts (BOOSTED_TREE exports as XGBoost Booster files).
# The GCS bucket must be in the SAME region as your BigQuery dataset.
try:
    run_ddl(f'''
    EXPORT MODEL `{PROJECT_ID}.ml_models.doc_classifier_v2`
    OPTIONS(URI = 'gs://{PROJECT_ID}-models/doc_classifier/v2/')
    ''')
    print('Model exported to GCS')
except Exception as e:
    print(f'Export requires a GCS bucket in the same region: {e}')

In [ ]:
# List the exported artifacts (XGBoost booster + metadata).
# Uses IPython ! magic so {PROJECT_ID} is substituted from the Python namespace.
# (A %%bash cell would NOT see the Python variable and would leave ${PROJECT_ID} empty.)
!gsutil ls -r gs://{PROJECT_ID}-models/doc_classifier/v2/ || echo "Create the bucket first: gsutil mb -l us-central1 gs://{PROJECT_ID}-models/"

## Exercise 3: bigframes Exploration

**Difficulty:** Easy

Read `document_features` with `bpd.read_gbq()`. Groupby, agg, print stats.

1. `pip install bigframes`
2. `bpd.read_gbq()` to load data
3. groupby + agg + peek()

In [ ]:
%%bash
pip install -q bigframes

In [ ]:
import bigframes.pandas as bpd

bpd.options.bigquery.project = PROJECT_ID
bpd.options.bigquery.location = 'US'

# Read DocuMind data as a BigQuery-backed DataFrame (data never leaves the warehouse).
docs = bpd.read_gbq(f'{PROJECT_ID}.rag_data.document_features')

# Pandas operations executed at BigQuery scale
stats = (
    docs.groupby('document_type')
    .agg({
        'page_count': 'mean',
        'processing_cost_usd': 'sum',
        'doc_id': 'count'
    })
    .rename(columns={'doc_id': 'doc_count'})
    .sort_values('doc_count', ascending=False)
)

print('=== Document Stats ===')
print(stats.peek(10))

# Inspect the SQL bigframes generated under the hood
print('\n=== Generated SQL ===')
print(stats.sql)

## Exercise 4: bigframes ML Pipeline

**Difficulty:** Medium

Build a Pipeline with StandardScaler + XGBClassifier. Train. Score. Save.

1. ColumnTransformer for scaling + encoding
2. Pipeline with preprocessing + classifier
3. fit, score, to_gbq

In [ ]:
from bigframes.ml.pipeline import Pipeline
from bigframes.ml.compose import ColumnTransformer
from bigframes.ml.preprocessing import StandardScaler, OneHotEncoder
from bigframes.ml.ensemble import XGBClassifier
from bigframes.ml.model_selection import train_test_split

# Prepare data
X = docs.drop(columns=['document_type', 'doc_id', 'title'])
y = docs[['document_type']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Preprocessing: scale numerics + one-hot encode the categorical
preprocessor = ColumnTransformer([
    ('scale', StandardScaler(),
     ['page_count', 'total_word_count', 'file_size_mb']),
    ('encode', OneHotEncoder(),
     ['content_type'])
])

pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', XGBClassifier())
])

# Train (bigframes compiles this into a BQML CREATE MODEL)
pipeline.fit(X_train, y_train)
score = pipeline.score(X_test, y_test)
print('=== bigframes ML Score ===')
print(score.to_pandas())

# Persist the fitted pipeline as a BQML model
pipeline.to_gbq(f'{PROJECT_ID}.ml_models.doc_classifier_bf', replace=True)
print('Pipeline saved as BQML model')

## Exercise 5: TRANSFORM with 5+ Functions

**Difficulty:** Medium

Create a model with STANDARD_SCALER, QUANTILE_BUCKETIZE, FEATURE_CROSS, LOG, SAFE_DIVIDE.

1. TRANSFORM clause with 5+ preprocessing functions
2. Train BOOSTED_TREE_CLASSIFIER
3. ML.GLOBAL_EXPLAIN to check engineered features

In [ ]:
# The TRANSFORM clause bakes feature engineering INTO the model, so it is
# applied automatically at prediction time (train/serve skew disappears).
run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.doc_classifier_transform`
TRANSFORM (
  ML.STANDARD_SCALER(total_word_count) OVER() AS words_scaled,
  ML.MIN_MAX_SCALER(page_count) OVER() AS pages_scaled,
  ML.QUANTILE_BUCKETIZE(file_size_mb, 5) OVER() AS size_bucket,
  ML.FEATURE_CROSS(STRUCT(content_type, CAST(page_count > 10 AS STRING))) AS type_length,
  LOG(total_word_count + 1) AS log_words,
  SAFE_DIVIDE(chunk_count, page_count) AS chunks_per_page,
  document_type
)
OPTIONS (
  model_type = 'BOOSTED_TREE_CLASSIFIER',
  input_label_cols = ['document_type'],
  enable_global_explain = TRUE,
  model_registry = 'vertex_ai',
  vertex_ai_model_id = 'documind_classifier_transform'
) AS
SELECT * FROM `{PROJECT_ID}.rag_data.document_features`
''')
print('Model with TRANSFORM trained and registered')

# Attribution per engineered feature
print('\n=== Transform Model Feature Importance ===')
print(run_query(f'SELECT * FROM ML.GLOBAL_EXPLAIN(MODEL `{PROJECT_ID}.ml_models.doc_classifier_transform`)'))

## Exercise 6: Window Features

**Difficulty:** Medium

Create `doc_time_features` with AVG OVER, LAG, RANK. Use in a model.

1. Rolling average, lag, rank window functions
2. CREATE TABLE from window query
3. Join features into model training data

In [ ]:
# Rolling average (3-row window), previous-day lag, and daily rank.
try:
    run_ddl(f'''
    CREATE OR REPLACE TABLE `{PROJECT_ID}.rag_data.doc_time_features` AS
    WITH daily AS (
      SELECT
        doc_id,
        DATE_ADD(DATE '2026-01-01', INTERVAL CAST(SUBSTR(doc_id, 2) AS INT64) DAY) AS query_date,
        page_count AS daily_metric
      FROM `{PROJECT_ID}.rag_data.document_features`
    )
    SELECT
      doc_id, query_date, daily_metric,
      AVG(daily_metric) OVER (
        PARTITION BY doc_id ORDER BY query_date
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
      ) AS rolling_avg_3d,
      LAG(daily_metric, 1) OVER (
        PARTITION BY doc_id ORDER BY query_date
      ) AS prev_day,
      RANK() OVER (
        PARTITION BY query_date ORDER BY daily_metric DESC
      ) AS daily_rank
    FROM daily
    ''')
    print('Time-based features created')
    print(run_query(f'SELECT * FROM `{PROJECT_ID}.rag_data.doc_time_features` LIMIT 5'))
except Exception as e:
    print(f'Note: {e}')

## Exercise 7: Deploy to Endpoint

**Difficulty:** Challenge

Deploy the registered model to a Vertex AI endpoint. Send a prediction. Check latency.

1. `aiplatform.Model.list()` to find the model
2. `Endpoint.create()` + `model.deploy()`
3. `endpoint.predict()` with a test instance

> Deploying keeps a machine warm (~$55/month for `e2-standard-2`). **Undeploy when done** — the cleanup cell at the end handles this.

In [ ]:
from google.cloud import aiplatform
import time

aiplatform.init(project=PROJECT_ID, location=LOCATION)

# Find the model registered in Exercise 1
models = aiplatform.Model.list(filter='display_name="documind_doc_classifier"')
if not models:
    raise RuntimeError("Model 'documind_doc_classifier' not found -- run Exercise 1 (CREATE MODEL ... model_registry='vertex_ai') first.")
model = models[0]
print(f'Found model: {model.resource_name}')

# Create an endpoint and deploy (this can take ~10-15 min)
endpoint = aiplatform.Endpoint.create(display_name='documind-classifier-endpoint')
model.deploy(
    endpoint=endpoint,
    machine_type='e2-standard-2',
    min_replica_count=1,
    max_replica_count=3)
print('Deployed. Sending a test prediction...')

# Real-time prediction + crude latency check
start = time.time()
result = endpoint.predict(instances=[{
    'page_count': 12,
    'chunk_count': 24,
    'total_word_count': 8500,
    'file_size_mb': 2.5,
    'content_type': 'pdf',
    'avg_chunk_size': 354.0
}])
latency_ms = (time.time() - start) * 1000
print(result.predictions)
print(f'Round-trip latency: {latency_ms:.0f} ms')

## Exercise 8: Full Production Path

**Difficulty:** Challenge

Train with TRANSFORM + registry. Export. Deploy. Predict. Compare batch vs endpoint cost.

1. CREATE MODEL with TRANSFORM + model_registry
2. EXPORT MODEL to GCS
3. Deploy to endpoint and predict
4. Compare: ML.PREDICT cost vs endpoint cost for 500 docs/day

Steps 1–3 reuse the models you already built above (Exercise 5's TRANSFORM+registry model, Exercise 2's export, Exercise 7's endpoint). The cell below closes the loop with the batch-vs-real-time cost comparison.

In [ ]:
# Batch (ML.PREDICT) runs inside BigQuery and is billed per TB scanned -
# for 500 small docs/day it is effectively pennies.
# A deployed endpoint bills for a warm machine 24x7 whether or not you call it.

DOCS_PER_DAY = 500

# Endpoint: 1x e2-standard-2 held warm ~ $55/month (Vertex AI serving list price)
endpoint_usd_month = 55.0

# Batch: ML.PREDICT over a tiny feature table - well under BigQuery's 1 TB/month
# free tier, so treat as ~ $0. Show a nominal on-demand estimate instead.
batch_bytes_per_run = 50 * 1024 * 1024          # ~50 MB scanned per daily run
bytes_per_month = batch_bytes_per_run * 30
tb_per_month = bytes_per_month / (1024 ** 4)
batch_usd_month = tb_per_month * 6.25           # $6.25 / TB on-demand

print(f'Workload: {DOCS_PER_DAY} docs/day\n')
print(f'Batch  (ML.PREDICT):  ${batch_usd_month:>7.4f}/mo  (Rs {batch_usd_month * USD_INR:>8.2f})')
print(f'Endpoint (real-time): ${endpoint_usd_month:>7.2f}/mo  (Rs {endpoint_usd_month * USD_INR:>8.2f})')
print('\nVerdict: use ML.PREDICT for scheduled/batch scoring;')
print('reserve an endpoint only when you truly need sub-second online latency.')

### Cleanup — undeploy the endpoint

Run this after Exercises 7/8 so the warm machine stops billing.

In [ ]:
try:
    endpoint.undeploy_all()
    endpoint.delete()
    print('Endpoint undeployed and deleted - no more serving charges.')
except Exception as e:
    print(f'Nothing to clean up (or already deleted): {e}')